#### Modules

In [34]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import tensorflow as tf
from keras import layers, models
from keras.callbacks import EarlyStopping
from sklearn.metrics import confusion_matrix, classification_report

#### Train and testing splits 

Stratify ensures training, validation and testing datasets have the exact same proportion of crack and uncrack as the original dataset

In [35]:
# Load datasets
df_walls = pd.read_pickle('wall_data.pkl')
df_decks = pd.read_pickle('deck_data.pkl')

# Split the Walls DataFrame (15% of total wall data for validation, hence 0.15 / 0.85 ≈ 0.176 for second split)
walls, walls_te = train_test_split(df_walls, test_size=0.2, random_state=42, stratify=df_walls['Label'])
walls_tr, walls_val = train_test_split(walls, test_size=0.176, random_state=42, stratify=walls['Label'])

# Split the Decks DataFrame
decks, decks_te = train_test_split(df_decks, test_size=0.2, random_state=42, stratify=df_decks['Label'])
decks_tr, decks_val = train_test_split(decks, test_size=0.176, random_state=42, stratify=decks['Label'])

#### CNN Architecture

In [37]:
def cnn(input_shape):
    
    model = models.Sequential()

    # Filter 1
    model.add(layers.Conv2D(32, (3, 3), activation='relu', input_shape=input_shape))
    model.add(layers.MaxPooling2D((2, 2)))

    # Filter 2
    model.add(layers.Conv2D(64, (3, 3), activation='relu'))
    model.add(layers.MaxPooling2D((2, 2)))

    # Filter 3
    model.add(layers.Conv2D(128, (3, 3), activation='relu'))
    model.add(layers.MaxPooling2D((2, 2)))

    # Set up input for neural network for classification
    model.add(layers.Flatten())

    # Layer 1 (Dropout to prevent overfitting)
    model.add(layers.Dense(128, activation='relu'))
    model.add(layers.Dropout(0.5))
    
    # Layer 2
    model.add(layers.Dense(64, activation='relu'))
    model.add(layers.Dropout(0.5))
    
    # Output layer - use sigmoid to output a probability between 0.0 (Non-Cracked) and 1.0 (Cracked)
    model.add(layers.Dense(1, activation='sigmoid'))

    # Adam is a smart optimizer, Binary Crossentropy is standard for 2 classes
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    
    return model

# Set up EarlyStopping callback to prevent overfitting
early_stopper = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# Define label
label_map = {'Non-Cracked': 0.0, 'Cracked': 1.0}

#### Data Generator (Memory Efficient)

Instead of loading all images into memory at once, we use a generator that loads images in batches during training.

In [44]:
# Custom data generator to load images in batches (memory efficient)
class DataGenerator(tf.keras.utils.Sequence):
    def __init__(self, dataframe, batch_size=32, shuffle=True):
        self.dataframe = dataframe.reset_index(drop=True)
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.indexes = np.arange(len(self.dataframe))
        self.on_epoch_end()
    
    def __len__(self):
        # Number of batches per epoch
        return int(np.ceil(len(self.dataframe) / self.batch_size))
    
    def __getitem__(self, index):
        # Get batch indexes
        batch_indexes = self.indexes[index * self.batch_size:(index + 1) * self.batch_size]
        
        # Generate batch data
        X = np.stack(self.dataframe.iloc[batch_indexes]['ImageData'].values) / 255.0
        y = self.dataframe.iloc[batch_indexes]['Label'].map(label_map).values.astype(np.float32)
        
        return X, y
    
    def on_epoch_end(self):
        # Shuffle indexes after each epoch
        if self.shuffle:
            np.random.shuffle(self.indexes)

#### Training phase

In [40]:
# Define model input shape
format = (256, 256, 3)
model_walls = cnn(format)
model_decks = cnn(format)

In [ ]:
# Create data generators for Walls dataset (memory efficient)
train_gen_walls = DataGenerator(walls_tr, batch_size=32, shuffle=True)
val_gen_walls = DataGenerator(walls_val, batch_size=32, shuffle=False)
test_gen_walls = DataGenerator(walls_te, batch_size=32, shuffle=False)

In [ ]:
# Create data generators for Decks dataset (memory efficient)
train_gen_decks = DataGenerator(decks_tr, batch_size=32, shuffle=True)
val_gen_decks = DataGenerator(decks_val, batch_size=32, shuffle=False)
test_gen_decks = DataGenerator(decks_te, batch_size=32, shuffle=False)

In [ ]:
# Train using data generators (memory efficient)
history_walls = model_walls.fit(
    train_gen_walls,
    epochs=300,
    validation_data=val_gen_walls,
    callbacks=[early_stopper])

Epoch 1/300
 34/370 ━━━━━━━━━━━━━━━━━━━━ 3:03 547ms/step - accuracy: 0.7318 - loss: 0.9686

KeyboardInterrupt: 